In [6]:
from google.cloud import bigquery

client = bigquery.Client(project="students-group3")

query_movies = """
SELECT *
FROM `students-group3.MovieData.movies_cleaned`
"""

df_movies = client.query(query_movies).to_dataframe()

df_movies.head()


,movieId,title,genres_list
0,117867,'71 (2014),"[Action, Drama, Thriller, War]"
1,97757,'Hellboy': The Seeds of Creation (2004),"[Action, Adventure, Comedy, Documentary, Fantasy]"
2,26564,'Round Midnight (1986),"[Drama, Musical]"
3,779,'Til There Was You (1997),"[Drama, Romance]"
4,2072,"'burbs, The (1989)",[Comedy]


In [8]:
query_ratings = """
SELECT *
FROM `students-group3.MovieData.ratings_cleaned`
"""

df_ratings = client.query(query_ratings).to_dataframe()
df_ratings.head()


,userId,movieId,rating
0,1,204,0.5
1,1,256,0.5
2,1,277,0.5
3,1,719,0.5
4,1,45950,0.5


## Encodogae des genres

In [11]:
# Extraire tous les genres uniques
unique_genres = sorted(
    set(genre for genres in df_movies["genres_list"] for genre in genres)
)

unique_genres


['(no genres listed)',
 'Action',
 'Adventure',
 'Animation',
 'Children',
 'Comedy',
 'Crime',
 'Documentary',
 'Drama',
 'Fantasy',
 'Film-Noir',
 'Horror',
 'IMAX',
 'Musical',
 'Mystery',
 'Romance',
 'Sci-Fi',
 'Thriller',
 'War',
 'Western']

In [23]:
# Copier le DataFrame
df_movies_encoded = df_movies.copy()

# Nettoyer les noms de genres pour BigQuery
clean_genres = [g.replace(" ", "_").replace("(", "").replace(")", "") for g in unique_genres]

# Multi-Hot Encoding avec noms propres
for orig, clean in zip(unique_genres, clean_genres):
    df_movies_encoded[clean] = df_movies_encoded["genres_list"].apply(lambda x: int(orig in x))

df_movies_encoded.drop(columns=["genres_list"], inplace=True)

# Vérifier
df_movies_encoded.head()


,movieId,title,no_genres_listed,Action,Adventure,Animation,Children,Comedy,Crime,Documentary,...,Film-Noir,Horror,IMAX,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
0,117867,'71 (2014),0,1,0,0,0,0,0,0,...,0,0,0,0,0,0,0,1,1,0
1,97757,'Hellboy': The Seeds of Creation (2004),0,1,1,0,0,1,0,1,...,0,0,0,0,0,0,0,0,0,0
2,26564,'Round Midnight (1986),0,0,0,0,0,0,0,0,...,0,0,0,1,0,0,0,0,0,0
3,779,'Til There Was You (1997),0,0,0,0,0,0,0,0,...,0,0,0,0,0,1,0,0,0,0
4,2072,"'burbs, The (1989)",0,0,0,0,0,1,0,0,...,0,0,0,0,0,0,0,0,0,0


In [24]:
from pandas_gbq import to_gbq

to_gbq(
    df_movies_encoded,
    "MovieData.movies_encoded",
    project_id="students-group3",
    if_exists="replace"
)


100%|██████████| 1/1 [00:00<00:00, 5526.09it/s]
